# Airbridge Entry API - 신규 앱 온보딩 노트북

새로운 앱을 Airbridge Entry API에 연동할 때 사용하는 노트북입니다.  
위에서 아래로 순서대로 실행하면 됩니다.

### 이 노트북이 하는 일
1. 앱 정보 입력
2. 이벤트 목록 조회 (Snowflake 또는 CSV)
3. 이벤트 매핑 확인 (기본 커머스 매핑과 비교)
4. 매핑 수정 (필요시)
5. config 저장
6. 데이터 추출 테스트
7. D3 Purchase/Churn 모델 학습
8. 서버에 모델 업로드
9. 고객사 안내사항 출력

### 소요 시간
- 이벤트 매핑이 기본값 그대로 맞으면: ~10분
- 커스텀 매핑 필요하면: ~30분

### 사전 준비
- Airbridge 대시보드에서 앱의 `app_id` 확인
- 서버 URL 확인 (Render 또는 로컬)
- Python 환경 (pandas, scikit-learn, requests)

---
## Step 1: 앱 정보 입력

연동할 앱의 기본 정보를 입력합니다.

- **APP_ID**: Airbridge 대시보드에서 확인할 수 있는 앱 고유 번호 (Settings → App Info)
- **APP_NAME**: 서버에서 모델/config를 구분하는 이름 (영문, 소문자, 공백 없이)
- **SERVER_URL**: Entry API 서버 주소

In [ ]:
APP_ID = 30778  # ← 새 앱의 Airbridge app_id
APP_NAME = "ablog"  # ← 서버에서 사용할 앱 이름
SERVER_URL = "https://airbridge-entry-api-prototype.onrender.com"

print(f"앱 ID: {APP_ID}")
print(f"앱 이름: {APP_NAME}")
print(f"서버: {SERVER_URL}")

---
## Step 2: 이벤트 목록 조회

Snowflake에서 이 앱의 이벤트 종류를 조회합니다.  
어떤 이벤트가 실제로 태깅되어 들어오고 있는지 확인하는 단계입니다.

프로토타입에서는 Snowflake 대신 CSV로 대체합니다.  
실서비스 연동 시에는 주석 처리된 Snowflake 코드를 사용하세요.

In [ ]:
import pandas as pd
import json

# 실서비스: Snowflake에서 조회
# import snowflake.connector
# conn = snowflake.connector.connect(...)
# df_events = pd.read_sql("""
#     SELECT DATA__EVENTDATA__GOAL__CATEGORY AS goal_category, COUNT(*) AS cnt
#     FROM AIRBRIDGE.PUBLIC.INTERNAL_MOBILE_EVENTS
#     WHERE DATA__APP__APPID = {APP_ID}
#     GROUP BY 1 ORDER BY cnt DESC LIMIT 30
# """, conn)

# 프로토타입: CSV에서 로드
df_events = pd.read_csv('../query_and_sample/queryC2.csv')
print(f"이 앱의 주요 이벤트:")
print(df_events.head(15).to_string(index=False))

---
## Step 3: 이벤트 매핑 확인

기본 커머스 매핑(`configs/default_commerce.json`)을 로드하고,  
이 앱의 실제 이벤트와 일치하는지 확인합니다.

- ✅ : 실제 데이터에 해당 이벤트가 존재
- ❌ 확인필요 : 데이터에 없거나 이벤트 이름이 다를 수 있음

모든 매핑이 맞으면 Step 5로 건너뛰어도 됩니다.  
틀린 게 있으면 Step 4에서 수정하세요.

In [ ]:
# 기본 커머스 매핑 로드
with open('../configs/default_commerce.json') as f:
    config = json.load(f)

config['app_id'] = APP_ID
config['app_name'] = APP_NAME

print("=== 기본 커머스 이벤트 매핑 ===")
for our_name, airbridge_name in config['event_mapping'].items():
    # 실제 데이터에 있는지 체크
    exists = airbridge_name in df_events['GOAL_CATEGORY'].values if 'GOAL_CATEGORY' in df_events.columns else '확인필요'
    status = "✅" if exists == True else "❌ 확인필요"
    print(f"  {our_name:20s} → {airbridge_name:45s} {status}")

print()
print("매핑이 맞으면 다음 셀로 넘어가세요.")
print("틀린 게 있으면 아래에서 수정하세요:")

---
## Step 4: 매핑 수정 (필요시)

Step 3에서 ❌ 표시된 이벤트가 있으면 여기서 수정합니다.  
앱마다 이벤트 이름이 다를 수 있으니, Airbridge 대시보드 또는 Step 2 결과를 보고 맞는 이벤트명으로 바꿔주세요.

수정이 필요 없으면 이 셀은 그대로 실행만 하면 됩니다.

In [ ]:
# 매핑 수정이 필요하면 여기서 수정
# 예: config['event_mapping']['purchase'] = 'order_complete'

# 확인
print("최종 매핑:")
for k, v in config['event_mapping'].items():
    print(f"  {k}: {v}")

---
## Step 5: config 저장

확정된 이벤트 매핑을 `configs/{APP_NAME}.json`으로 저장합니다.  
이 파일은 데이터 파이프라인과 서버에서 참조합니다.

In [ ]:
import os
os.makedirs('../configs', exist_ok=True)
config_path = f'../configs/{APP_NAME}.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(f"✅ {config_path} 저장 완료!")

---
## Step 6: 데이터 추출 테스트

매핑이 맞는지 소량 데이터로 테스트합니다.  
실서비스에서는 Snowflake에서 최근 3일 데이터를 추출하여 feature 생성이 정상적인지 확인합니다.

프로토타입에서는 기존 샘플 데이터로 모델을 학습합니다.

In [ ]:
# 실서비스: Snowflake에서 소량 추출
# df_test = pd.read_sql("""
#     SELECT * FROM AIRBRIDGE.PUBLIC.INTERNAL_MOBILE_EVENTS
#     WHERE DATA__APP__APPID = {APP_ID}
#     AND DATA__TIMESTAMP >= DATEADD('day', -3, CURRENT_DATE())
#     LIMIT 1000
# """, conn)
# print(f"추출 건수: {len(df_test)}")
# print(df_test['DATA__EVENTDATA__GOAL__CATEGORY'].value_counts().head(10))

# 프로토타입: 기존 sample 데이터 사용
print("데이터 추출 테스트는 실서비스 Snowflake 연동 후 가능합니다.")
print("프로토타입에서는 기존 sample 데이터로 모델을 학습합니다.")

---
## Step 7: D3 Purchase/Churn 모델 학습

25개 feature(UA 15개 + In-app 10개)로 두 가지 모델을 학습합니다:

| 모델 | 예측 대상 | 용도 |
|------|----------|------|
| D3 Purchase | 3일 내 구매 확률 | 고가치 유저 식별 |
| D3 Churn | 3일 내 이탈 확률 | 이탈 위험 유저 식별 |

두 모델 모두 **전체 유저**로 학습합니다.  
구매/이탈 예측은 trigger 배정과 무관하므로 is_random 여부와 상관없이 모든 유저 데이터를 사용할 수 있습니다.

모델은 `models/{APP_NAME}/` 디렉토리에 pkl 파일로 저장됩니다.

In [ ]:
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Feature definitions (must match server/predict.py)
UA_FEATURES = [
    'trackinglink_count', 'DA_count', 'SA_count',
    'unique_channel_count', 'channel_entropy', 'last_touch_is_da',
    'latency', 'recency', 'recent_touch_pressure',
    'touch_per_latency_hour', 'last1h_touch_count', 'recent_24h_ratio',
    'click_ratio', 'impression_count', 'is_single_touch_install'
]

INAPP_FEATURES = [
    'product_viewed_count', 'user_signin', 'product_addedtocart',
    'deeplink_open', 'home_viewed', 'addtowishlist',
    'onboarding', 'user_signup', 'total_events', 'n_event_types'
]

ALL_FEATURES = UA_FEATURES + INAPP_FEATURES
print(f"Feature 수: UA {len(UA_FEATURES)}개 + In-app {len(INAPP_FEATURES)}개 = 전체 {len(ALL_FEATURES)}개")

# Model output directory
model_dir = f'../models/{APP_NAME}'
os.makedirs(model_dir, exist_ok=True)

# Load data
df = pd.read_csv('../data/weekly_data.csv')
if 'app_id' in df.columns:
    df = df[df['app_id'] == APP_NAME].reset_index(drop=True)

print(f"학습 데이터: {len(df)}명")
print(f"D3 구매율: {df['d3_purchase'].mean():.1%}")
print(f"D3 이탈율: {df['d3_churn'].mean():.1%}")

X_all = df[ALL_FEATURES].values

# --- D3 Purchase Model ---
print("\n=== D3 Purchase 모델 학습 ===")
y_purchase = df['d3_purchase'].values
purchase_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
purchase_model.fit(X_all, y_purchase)

scores_purchase = cross_val_score(purchase_model, X_all, y_purchase, cv=5, scoring='roc_auc')
print(f"D3 Purchase AUC: {scores_purchase.mean():.3f} (±{scores_purchase.std():.3f})")

purchase_data = {'model': purchase_model, 'feature_names': ALL_FEATURES}
with open(f'{model_dir}/d3_purchase_model.pkl', 'wb') as f:
    pickle.dump(purchase_data, f)
print(f"{model_dir}/d3_purchase_model.pkl 저장 완료!")

# --- D3 Churn Model ---
print("\n=== D3 Churn 모델 학습 ===")
y_churn = df['d3_churn'].values
churn_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
churn_model.fit(X_all, y_churn)

scores_churn = cross_val_score(churn_model, X_all, y_churn, cv=5, scoring='roc_auc')
print(f"D3 Churn AUC: {scores_churn.mean():.3f} (±{scores_churn.std():.3f})")

churn_data = {'model': churn_model, 'feature_names': ALL_FEATURES}
with open(f'{model_dir}/d3_churn_model.pkl', 'wb') as f:
    pickle.dump(churn_data, f)
print(f"{model_dir}/d3_churn_model.pkl 저장 완료!")

# --- Summary ---
print("\n=== 학습 결과 요약 ===")
print(f"  D3 Purchase AUC: {scores_purchase.mean():.3f}  (0.5=동전던지기, 0.8+=실무에서 쓸만함)")
print(f"  D3 Churn AUC:    {scores_churn.mean():.3f}")
print(f"  학습 데이터:      {len(df)}명 (전체 유저)")
print(f"  저장 위치:        {model_dir}/")

---
## Step 8: 서버에 업로드

학습된 모델 파일을 서버에 업로드합니다.  
서버가 자동으로 모델을 로드하므로 재배포 없이 바로 반영됩니다.

CATE 모델이 아직 없기 때문에 **exploration 모드**로 시작합니다:
- 모든 유저에게 trigger를 랜덤 배정
- D3 Purchase/Churn 확률은 바로 제공
- CATE 모델은 RCT 데이터가 충분히 쌓인 후 `weekly_training.ipynb`에서 학습

In [ ]:
import requests

model_files = [
    f"../models/{APP_NAME}/d3_purchase_model.pkl",
    f"../models/{APP_NAME}/d3_churn_model.pkl",
]

print(f"=== 서버에 모델 업로드 ({APP_NAME}) ===")
for filepath in model_files:
    filename = os.path.basename(filepath)
    if not os.path.exists(filepath):
        print(f"  [SKIP] {filename}")
        continue
    try:
        with open(filepath, "rb") as f:
            resp = requests.post(f"{SERVER_URL}/v1/models/{APP_NAME}/upload", files={"file": (filename, f)})
        if resp.status_code == 200:
            print(f"  [OK] {filename} → {resp.json()['mode']}")
        else:
            print(f"  [ERROR] {resp.status_code}: {resp.text}")
    except requests.ConnectionError:
        print(f"  [ERROR] 서버 연결 실패: {SERVER_URL}")
        break

print()
print("✅ 서버가 exploration 모드로 시작됩니다.")
print("   → 모든 유저에게 trigger를 랜덤 배정")
print("   → D3 Purchase/Churn 확률은 바로 제공")
print("   → CATE 모델은 RCT 데이터 충분히 쌓인 후 weekly_training에서 학습")

---
## Step 9: 고객사 안내사항

고객사에 전달해야 할 기술 연동 가이드입니다.  
아래 출력 내용을 복사하여 고객사 담당자에게 전달하세요.

포함 내용:
1. Airbridge SDK 이벤트 태깅 확인사항
2. `entry_modal_clicked` 이벤트 태깅 코드 (iOS/Android)
3. 모달 4종 디자인 가이드
4. API 연동 엔드포인트
5. 서비스 타임라인 (exploration → optimized)

In [ ]:
print(f"""
{'='*60}
고객사 안내사항 ({APP_NAME})
{'='*60}

1. Airbridge SDK 이벤트 태깅
   - 기존 이벤트 (product.viewed, signin 등)가 태깅되어 있는지 확인
   - 추가 태깅 필요: entry_modal_clicked

2. entry_modal_clicked 이벤트 태깅 코드:

   [iOS - Swift]
   Airbridge.trackEvent(
       category: "entry_modal_clicked",
       semanticAttributes: [:],
       customAttributes: [
           "trigger_type": "social_proof"  // price_appeal | social_proof | scarcity | novelty
       ]
   )

   [Android - Kotlin]
   Airbridge.trackEvent(
       "entry_modal_clicked",
       mapOf(),
       mapOf("trigger_type" to "social_proof")
   )

   ⚠️ Airbridge 대시보드에서 Event Taxonomy에 'entry_modal_clicked' 등록 필수!

3. 모달 4개 디자인 (포맷 동일하게):
   - Price Appeal: 할인/쿠폰 강조
   - Social Proof: 인기/랭킹 강조
   - Scarcity: 한정/긴급성 강조
   - Novelty: 신상품/트렌드 강조

4. API 연동:
   POST {SERVER_URL}/v1/entry/predict
   Body: {{"app_id": "{APP_NAME}", "airbridge_uuid": "<유저UUID>"}}

5. 타임라인:
   - 지금: exploration 모드 (trigger 랜덤 배정 + D3 예측 제공)
   - 2-4주 후: 충분한 데이터 수집 → CATE 모델 학습 → optimized 모드 전환
""")